In [3]:
from pycaret.classification import setup, compare_models
import pandas as pd

print("PyCaret 설치 성공!")

PyCaret 설치 성공!


In [11]:
# from pycaret.nlp import *
import pycaret
print(pycaret.__version__)


3.3.2


In [19]:
# sof : Start of function ------------------------------------------------------ #
import os
import pandas as pd

def load_file_data(path):
    """
    주어진 경로 목록에 있는 텍스트 파일들을 읽어 DataFrame으로 반환합니다.

    매개변수
    ----------
    path : list of str
        읽을 파일들의 전체 경로가 담긴 리스트입니다.
        각 파일은 encoding='latin1' 방식으로 열립니다.

    반환값
    ----------
    pandas.DataFrame
        두 개의 컬럼을 가진 DataFrame을 반환합니다.
        - 'filename' : 파일명(확장자 제외)
        - 'opinion_text' : 파일의 전체 텍스트 내용

    참고
    ----------
    - path가 비어 있거나 리스트가 아닐 경우 ValueError를 발생시킵니다.
    - 파일이 존재하지 않을 경우 오류 메시지를 출력하고 해당 파일은 건너뜁니다.
    - 파일을 읽는 중 다른 예외가 발생하면 오류 메시지를 출력하고 해당 파일은 건너뜁니다.
    - 여러 텍스트 파일을 하나의 구조화된 데이터로 모아 분석할 때 유용합니다.
    """
    # Validation
    if not path:
        raise ValueError("path가 비어 있습니다. 파일 경로 리스트를 전달해야 합니다.")
    if not isinstance(path, list):
        raise TypeError("path는 list 타입이어야 합니다. 예: ['file1.txt', 'file2.txt']")

    data_list = []

    for file_ in path:
        # 1. 파일 경로에서 파일명 추출 (확장자 제외)
        filename = os.path.basename(file_).split('.')[0]

        # 2. 파일 읽기 (latin1 인코딩 유지)
        try:
            with open(file_, 'r', encoding='latin1') as f:
                text_content = f.read()

            # 3. 딕셔너리 형태로 리스트에 추가
            data_list.append({
                'filename': filename,
                'opinion_text': text_content
            })

        except FileNotFoundError:
            print(f"Error: {file_} not found.")
        except Exception as e:
            print(f"An error occurred reading {file_}: {e}")

    # 반복문 종료 후 DataFrame 생성
    document_df = pd.DataFrame(data_list)
    return document_df

# eof : End of Function --------------------------------------------------------- #

총 문서 개수: 51


In [20]:
import pandas as pd
import os
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, plot_model



In [21]:
# 1. 데이터 로드s using 함수
path = r'../data'
all_files = glob.glob(os.path.join(path, "*.data"))  

document_df = load_file_data(all_files)

# 결과 확인
print("총 문서 개수:", len(document_df))
# print(document_df.head())  # 앞부분 확인
# print(document_df['opinion_text'][0][:200])  # 첫 번째 문서의 앞 200자 출력

총 문서 개수: 51


In [ ]:
# 2. TF-IDF 벡터화
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(document_df['opinion_text'].astype(str))
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# 3. PyCaret Clustering 환경 설정
s = setup(data=tfidf_df, session_id=42, silent=True, verbose=False)

# 4. 여러 군집화 모델 생성 및 비교
kmeans = create_model('kmeans')
dbscan = create_model('dbscan')
hierarchical = create_model('hierarchical')

# 5. 각 모델 결과 확인
plot_model(kmeans, plot='cluster')
plot_model(dbscan, plot='cluster')
plot_model(hierarchical, plot='cluster')

# 6. 클러스터 할당
clustered_df = assign_model(kmeans)
print(clustered_df.head())

NameError: name 'df' is not defined